Reinserting Trackers 1str+2str - PV Paiva (32p + 64p → 96p)
=
Works on `PV Paiva - Eletromecânico.dwg` at a stage where the trackers are `EMF_Tracker_1PX32_North/South` (1 string) and `EMF_Tracker_1PX64_North/South` (2 strings) blocks. It pairs consecutive tracker blocks in the same column, and writes a script that replaces each pair with one `EMF_tracker_96p` and re-inserts the unpaired blocks with their original name.

**Workflow overview:**
1. Convert the DWG to DXF and extract all model space entities.
2. Look at the entity counts per block name.
3. Keep only the four tracker blocks.
4. Define the pairing functions.
5. Pair the blocks.
6. Write the reinsert script.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Paiva - Eletromecânico.dwg` in `C:\Users\Usuario\Desktop\projetos\paiva`. The DXF is written next to the DWG (and deleted afterwards). The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** trackers are block references (`INSERT`) placed directly in model space.
- **Block names** (exact, case-sensitive `==`): `EMF_Tracker_1PX32_North`, `EMF_Tracker_1PX32_South`, `EMF_Tracker_1PX64_North`, `EMF_Tracker_1PX64_South`.
- **Layers:** the 32p blocks are on layer `EMF_em_tracker_32p` (exact name) and the 64p blocks on another layer (`EMF_em_tracker_64p`). The layer of the upper block decides the Y correction of a pair, and the new block goes on that layer.
- **Geometry:** trackers run north-south. Blocks of the same column have the same X (within 0.1 after rounding to 2 decimals), and two blocks are paired when their Y gap is ≤ 66. The Y corrections (42.7655 with the 32p on top, 21.5015 with the 64p on top) are specific to these block definitions.
- **Rotation:** all trackers are at rotation 0 in this drawing; rotation is dropped and everything is re-inserted at 0.
- **Running the `.scr` in AutoCAD:** each line is a plain command-line `._insert <block> X,Y 1 1 0` (spaces act as Enter), so:
  - the blocks `EMF_tracker_96p` and the four original tracker blocks must already be defined in the drawing;
  - they must **not** have *Scale uniformly* on, because the line answers X scale, Y scale and rotation;
  - if they have attributes, set `ATTREQ = 0`, because the script gives no attribute values;
  - everything is inserted with rotation 0.
  - delete the four original tracker blocks first, otherwise they end up duplicated.

Loading the DWG entities
=
- Copies `PV Paiva - Eletromecânico.dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only processes that single file.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) converts it to DXF (ACAD2018) in ``output_folder`, which is the same as `dwg_folder``.
- `extract_data_from_dxf` reads every model space entity with `ezdxf` into a list of dicts (no DuckDB here). For block references it keeps the block name, insertion point, rotation, color and every attribute tag as its own key.
- Deletes the DXF and the temp copy of the DWG afterwards.
- `pd` comes from the `*` import of `DWG_to_DuckDB_ETL`. `math` is not used.

In [2]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


Entity overview
=
Converts the entities into a DataFrame and prints `describe()`, `info()` and the count per block name. In the saved run the drawing has only trackers: 3025 `1PX64_South`, 1946 `1PX64_North`, 1347 `1PX32_North` and 1078 `1PX32_South`.

In [3]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y  rotation
count   7396.000000    7396.000000    7396.0
mean   35046.123692  128103.163453       0.0
std      598.040467     966.183745       0.0
min    33667.550000  126018.800025       0.0
25%    34601.690000  127315.224625       0.0
50%    34979.686308  128157.605775       0.0
75%    35464.785992  128982.236025       0.0
max    36625.890000  129774.083025       0.0
<class 'pandas.DataFrame'>
RangeIndex: 7396 entries, 0 to 7395
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      7396 non-null   str    
 1   layer     7396 non-null   str    
 2   handle    7396 non-null   str    
 3   name      7396 non-null   str    
 4   X         7396 non-null   float64
 5   Y         7396 non-null   float64
 6   rotation  7396 non-null   int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 780.2 KB
None
name
EMF_Tracker_1PX64_South    3025
EMF_Tracker_1PX64_North    1946
EMF_Tracker_1PX

Selecting the tracker blocks
=
- Keeps the four `EMF_Tracker_1PX32/64_North/South` blocks.
- Drops `handle`, `type` and `rotation` (everything is re-inserted at rotation 0).

In [6]:
mask = (ELM_df['name'] == 'EMF_Tracker_1PX32_North') | (ELM_df['name'] == 'EMF_Tracker_1PX64_North') | (ELM_df['name'] == 'EMF_Tracker_1PX32_South') | (ELM_df['name'] == 'EMF_Tracker_1PX64_South')
tracker_df = ELM_df[mask].copy()

tracker_df.drop(columns=['handle', 'type', 'rotation'], inplace=True)

print(tracker_df)
print(tracker_df.describe())
print(tracker_df.info())
print(tracker_df['name'].value_counts())

                   layer                     name             X              Y
0     EMF_em_tracker_32p  EMF_Tracker_1PX32_North  35794.290000  129434.080025
1     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35926.591051  129114.316025
2     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35920.291051  129114.316025
3     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35913.991051  129114.316025
4     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35945.491051  129070.814965
...                  ...                      ...           ...            ...
7391  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34670.986150  126149.510111
7392  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34677.286150  126149.490602
7393  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34664.686150  126106.022575
7394  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34670.986150  126106.007111
7395  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34677.286150  126105.987602

[7396 rows x 4 columns]
                  X        

Pairing functions
=
**`group_structures(df, max_distance=66)`**
- Rounds `X` and `Y` to 2 decimals (this also changes the frame passed in) and sorts by `X` ascending, then `Y` descending: column by column, top to bottom. The docstring says "Y then X", which is not what the code does.
- `x_diff` / `y_diff` = difference to the previous block in that order.
- A new group starts when `x_diff > 0.1` (new column), when `|y_diff| > 66` (too far from the block above), or at the first row.
- Extra rule `is_64_32_pair`: a new group should also start when the block and the one above are a 64p and a 32p.
  - Warning: it looks for lowercase `64p` / `32p` in the names, but the names are `...1PX64...` / `...1PX32...`, so it never matches and the rule does nothing. If it did match, it would split exactly the 32p + 64p pairs this notebook wants to merge.
  - Result: pairing is by position only. Two 32p or two 64p blocks within 66 of each other also become one `EMF_tracker_96p`; check `Name-1` / `Name-2` in the output.
- Groups of 1 or 2 are kept as they are. Bigger groups are cut into consecutive pairs from the top, and an odd last block stays alone. The cut ignores the actual gaps inside the group.
- Notes:
  - `distance` is computed but not used.
  - The docstring mentions TAG boundaries and a 22 m default; neither is implemented (the default is 66).
  - Every call writes the sorted frame to `Desktop\teste.xlsx` (debug export).

**`create_group_positions(groups)`**
- One row per group with the **first (upper)** block's `X` / `Y`, the group size and `Layer-1` (plus `Layer-2` for pairs), and `Name-1` / `Name-2` (the block names).

In [10]:
def group_structures(df, max_distance=66):
    """
    Groups adjacent structures in pairs (2) while respecting:
    - TAG boundaries
    - Maximum distance constraint (22m default)
    - Sorted by Position Y then X
    """
    # Sort by Position Y (primary) and Position X (secondary)
    df['X'] = round(df['X'],2)
    df['Y'] = round(df['Y'],2)
    df = df.sort_values(['X', 'Y'], ascending=[True, False]).reset_index(drop=True)

    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = round(df['X'].diff(),2)
    df['y_diff'] = round(df['Y'].diff(),2)
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Helper function to check if it's a 64p/32p pair

    def is_64_32_pair(current_name, prev_name):
        if pd.isna(prev_name):
            return False
        has_64p = '64p' in str(current_name) or '64p' in str(prev_name)
        has_32p = '32p' in str(current_name) or '32p' in str(prev_name)
        return has_64p and has_32p and current_name != prev_name

    # Create group boundaries where:
    # 1. X is different
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    # 4. One structure is 64p and the other is 32p
    df['group_boundary'] = (
        (df['x_diff'] > 0.1) |
        (abs(df['y_diff']) > max_distance) |
        (df.index == 0) |
        (df.apply(lambda row: is_64_32_pair(row['name'], df.loc[row.name-1,'name'] if row.name > 0 else None), axis=1))
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        # Split groups larger than 2 into pairs
        size = len(group_df)
        if size <= 2:
            result_groups.append(group_df)

            continue

        n_pairs = size // 2
        for i in range(n_pairs):
            result_groups.append(group_df.iloc[i*2:i*2+2])


        if size % 2:  # Handle odd-sized groups
            result_groups.append(group_df.iloc[-1:])
    df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")
    return result_groups

def create_group_positions(groups):
    """Create the output DataFrame from groups (same as before)"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        if len(group_df) > 1:
            first_struct = group_df.iloc[0]
            second_struct = group_df.iloc[1]
            group_data.append({
                'Group_ID': i,
                'X': first_struct['X'],
                'Y': first_struct['Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer'],
                'Name-1': first_struct['name'],
                'Name-2': second_struct['name']
            })
        else:
            first_struct = group_df.iloc[0]
            group_data.append({
                'Group_ID': i,
                'X': first_struct['X'],
                'Y': first_struct['Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Name-1': first_struct['name']
            })
    return pd.DataFrame(group_data)

Pairing the blocks
=
- Runs the pairing on all four tracker blocks together.
- Exports the groups to `Desktop\teste1.xlsx` (debug) and prints the group sizes: 3988 single blocks and 1704 pairs in the saved run.

In [11]:
results = group_structures(tracker_df)
group_positions = create_group_positions(results)
group_positions.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")
print(group_positions['Group_Size'].value_counts())
print(group_positions)

Group_Size
1    3988
2    1704
Name: count, dtype: int64
      Group_ID         X          Y  Group_Size             Layer-1  \
0            1  33667.55  129034.80           2  EMF_em_tracker_32p   
1            2  33667.55  128909.06           1  EMF_em_tracker_64p   
2            3  33667.55  128823.02           1  EMF_em_tracker_64p   
3            4  33673.85  129034.80           2  EMF_em_tracker_32p   
4            5  33673.85  128909.06           1  EMF_em_tracker_64p   
...        ...       ...        ...         ...                 ...   
5687      5688  36600.69  129536.32           1  EMF_em_tracker_32p   
5688      5689  36606.99  129536.32           1  EMF_em_tracker_32p   
5689      5690  36613.29  129536.32           1  EMF_em_tracker_32p   
5690      5691  36619.59  129536.32           1  EMF_em_tracker_32p   
5691      5692  36625.89  129536.32           1  EMF_em_tracker_32p   

                 Layer-2                   Name-1                   Name-2  
0     EMF_em_

Reinsert script
=
- Single block → `._insert <original name> X,Y 1 1 0` at the same point.
- Pair → `._insert EMF_tracker_96p X,Y 1 1 0`:
  - upper block on layer `EMF_em_tracker_32p` → `Y − 42.7655`;
  - otherwise (64p on top) → `Y − 21.5015`.
  - The North / South distinction of the pair is lost: there is only one `EMF_tracker_96p` block.
- Exports the table to `...\Ecotechne\Separador de Strings\teste.xlsx` (debug; this is a different project folder).
- Writes the lines grouped by `Layer-1`, each group after `(command-s "._layer" "set" "<layer>" "")`, to `Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 1st+2str.scr` (`1st`, not `1str`).

In [13]:
correction_position_y_32_first = 42.7655
correction_position_y_64_first = 21.5015

group_positions['Script'] = np.where(
    group_positions['Group_Size'] == 1,
    group_positions.apply(
        lambda row: (
            f"._insert {row['Name-1']} {row['X']},{row['Y']} 1 1 0"
        ),
    axis=1
    ),
    np.where(
        group_positions['Layer-1'] == 'EMF_em_tracker_32p',
        group_positions.apply(
            lambda row: (
                f"._insert EMF_tracker_96p {row['X']},{row['Y']-correction_position_y_32_first} 1 1 0"
            ),
        axis=1
        ),
        group_positions.apply(
            lambda row: (
                f"._insert EMF_tracker_96p {row['X']},{row['Y']-correction_position_y_64_first} 1 1 0"
            ),
        axis=1
        )
    )
)

print(group_positions)

group_positions.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\teste.xlsx")
with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 1st+2str.scr", 'w') as f:
    for i in group_positions['Layer-1'].unique():
        mask = group_positions['Layer-1'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(group_positions[mask]['Script'].astype(str)) + '\n')

      Group_ID         X          Y  Group_Size             Layer-1  \
0            1  33667.55  129034.80           2  EMF_em_tracker_32p   
1            2  33667.55  128909.06           1  EMF_em_tracker_64p   
2            3  33667.55  128823.02           1  EMF_em_tracker_64p   
3            4  33673.85  129034.80           2  EMF_em_tracker_32p   
4            5  33673.85  128909.06           1  EMF_em_tracker_64p   
...        ...       ...        ...         ...                 ...   
5687      5688  36600.69  129536.32           1  EMF_em_tracker_32p   
5688      5689  36606.99  129536.32           1  EMF_em_tracker_32p   
5689      5690  36613.29  129536.32           1  EMF_em_tracker_32p   
5690      5691  36619.59  129536.32           1  EMF_em_tracker_32p   
5691      5692  36625.89  129536.32           1  EMF_em_tracker_32p   

                 Layer-2                   Name-1                   Name-2  \
0     EMF_em_tracker_32p  EMF_Tracker_1PX32_North  EMF_Tracker_1PX32_